In [ ]:
import requests
import re
from bs4 import BeautifulSoup
import time
import csv

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def save_to_csv(data, filename="thuviennhadat_raw.csv"):
    if not data:
        print("Không có dữ liệu để ghi CSV.")
        return
    desired_order = [
        "tieu_de",
        "gia",
        "dia_chi",
        "dien_tich_dat",   
        "phong_ngu",
        "phong_tam",       
        "so_tang",
        "phap_ly",
        "ngay_dang"
    ]

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=desired_order)
        writer.writeheader()

        for row in data:
            ordered_row = {key: row.get(key, "") for key in desired_order}
            writer.writerow(ordered_row)

    print(f"Đã ghi {len(data)} dòng vào file: {filename}")

def extract_so_tang(soup):
    """
    Tìm số tầng trong nội dung mô tả bằng một biểu thức Regex duy nhất.
    Trả về int (số tầng lớn nhất tìm được) hoặc None nếu không tìm thấy.
    """
    if not soup:
        return None

    # 1. THU THẬP VĂN BẢN ĐA DẠNG (Phần này rất tốt, giữ nguyên)
    texts = []
    
    # a) section mô tả nếu có
    desc = soup.find(id="description-section")
    if desc:
        texts.append(desc.get_text(" ", strip=True))

    # b) vùng chứa p theo class (nếu có)
    col = soup.find("div", class_="eleven wide column stackable")
    if col:
        texts.append(col.get_text(" ", strip=True))

    # c) fallback: tất cả các thẻ p (nếu không có class cụ thể)
    ps = soup.find_all("p")
    if ps:
        # Giới hạn số lượng p để tránh quá tải nếu trang có quá nhiều thẻ p rỗng
        texts.append(" ".join(p.get_text(" ", strip=True) for p in ps[:20]))

    # d) fallback toàn trang 
    texts.append(soup.get_text(" ", strip=True))

    full_text = " ".join(t for t in texts if t).lower()

    if not full_text:
        return None

    # 2. CHUẨN HOÁ VĂN BẢN
    # Chuẩn hoá dấu gạch (nếu có)
    full_text = full_text.replace("\u2013", " ").replace("\u2014", " ")
    # Chuẩn hoá tiếng Việt (Chuyển "tầng" thành "tang" để tìm kiếm không dấu luôn)
    full_text = full_text.replace("tầng", "tang") 
    
    # 3. LOGIC REGEX TỐI ƯU (Hợp nhất tất cả các kịch bản)
    
    # Pattern tìm kiếm:
    # Mục tiêu: Bắt số (tối đa 2 chữ số) đứng GẦN từ 'tang' (hoặc 'tầng' đã chuẩn hóa)
    # Rationale: 
    # (\d{1,2}): Nhóm 1: Bắt số 1 hoặc 2 chữ số (vd: 4, 15, 20)
    # \s*[\s\-\/]{0,3}: Tối đa 3 ký tự (khoảng trắng, gạch ngang, gạch chéo)
    # \b: Bắt buộc phải là ranh giới từ để tránh lỗi '7m2 tang'
    
    # Kịch bản 1: Số trước từ 'tang' (vd: 4 tầng, 4-tầng, 4/tang)
    pattern1 = r'(\d{1,2})\s*[\s\-\/]{0,3}tang\b'
    
    # Kịch bản 2: Số sau từ 'tang' (vd: tầng 4, tang-5)
    pattern2 = r'\btang\s*[\s\-\/]{0,3}(\d{1,2})' 
    
    combined_pattern = re.compile(f'{pattern1}|{pattern2}', re.IGNORECASE)
    
    matches = combined_pattern.findall(full_text)
    numbers_found = []
    
    for match in matches:
        # Match trả về tuple (ví dụ: ('4', '') cho pattern1, hoặc ('', '7') cho pattern2)
        try:
            number_str = match[0] if match[0] else match[1]
            if number_str:
                numbers_found.append(int(number_str))
        except ValueError:
            continue
            
    if not numbers_found:
        return None

    return max(numbers_found)

# -----------------------------
# 1) Danh sách khu vực được phép
# -----------------------------
ALLOWED_AREAS = {
    # Quận
    "hoàng mai", "long biên", "hà đông", "tây hồ",
    "nam từ liêm", "bắc từ liêm",

}

def detect_area_from_address(address):
    if not address:
        return None

    addr_lower = address.lower()

    for area in ALLOWED_AREAS:
        if area in addr_lower:
            return area

    return None


# -----------------------------
# 2) Hàm scrape chi tiết bài đăng (giữ nguyên)
# -----------------------------
def scrape_detail(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        res = requests.get(url, timeout=10, headers=headers)
        if res.status_code != 200:
            return None
        soup = BeautifulSoup(res.text, "html.parser")

        data = {}

        # Tiêu đề + địa chỉ
        title_block = soup.find("p", class_="text-truncate-2")
        if not title_block:
            return None
        tieu_de = title_block.get_text(strip=True)
        data["tieu_de"] = tieu_de
        data["dia_chi"] = tieu_de.split("tại", 1)[-1].strip() if "tại" in tieu_de.lower() else tieu_de

        area = detect_area_from_address(data["dia_chi"])
        if not area:
            return None 

        # Các thông tin khác
        for block in soup.find_all("p", class_="unit-name-style"):
            name = block.get_text(strip=True)
            value = block.find_next("span", class_="unit-value-style")
            value = value.get_text(strip=True) if value else None
            if "Mức giá" in name: data["gia"] = value
            elif "Diện tích" in name: data["dien_tich_dat"] = value
            elif "phòng ngủ" in name: data["phong_ngu"] = value
            elif "WC" in name: data["phong_tam"] = value
            elif "Pháp lý" in name: data["phap_ly"] = value

        ngay = soup.find("div", string=lambda x: x and "Ngày đăng" in x)
        if ngay:
            data["ngay_dang"] = ngay.find_next("div", class_="text-primary").get_text(strip=True)

        data["so_tang"] = extract_so_tang(soup)
        #data["khu_vuc"] = area.title()
        #data["url"] = url

        return data

    except Exception as e:
        return None


# -----------------------------
# 3) Hàm lấy link chi tiết từ 1 trang danh sách
# -----------------------------

driver = None

def init_driver():
    global driver
    if driver is None:
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_argument("--disable-gpu")
        options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        driver = webdriver.Chrome(options=options)
    return driver

def scrape_list_page(url: str):
    print(f"Đang crawl danh sách: {url}")
    driver = init_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.navigateTo.mb-mt-15 a"))
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")
        links = set()
        for a in soup.select("div.navigateTo.mb-mt-15 a[href*='pst']"):
            href = a.get("href")
            if href and href.startswith("/"):
                links.add("https://thuviennhadat.vn" + href)
        print(f"→ Tìm thấy {len(links)} link")
        return sorted(list(links))
    except Exception as e:
        print(f"Lỗi trang {url}: {e}")
        return []

# -----------------------------
# 4) Hàm crawl toàn bộ
# -----------------------------
from concurrent.futures import ThreadPoolExecutor, as_completed

def crawl_full(target_count=2500, batch_size=150):
    base_url = "https://thuviennhadat.vn/ban-nha-tai-dinh-cu-thanh-pho-ha-noi?cIds=6,5,4,3"
    print(f"Mục tiêu: {target_count} bài hợp lệ → Bắt đầu crawl...\n")

    all_results = []
    page = 1
    processed_links = 0  # đếm tổng link đã xử lý (dù hợp lệ hay không)

    while len(all_results) < target_count and page <= 1000:
        url = base_url if page == 1 else f"{base_url}&trang={page}"
        
        # Bước 1: Lấy link trang hiện tại
        links = scrape_list_page(url)
        if not links:
            print("Không có link nào, bỏ qua trang này")
            page += 1
            continue

        print(f"Trang {page} → {len(links)} link mới → Bắt đầu xử lý song song...")

        # Bước 2: Xử lý ngay batch này bằng đa luồng
        batch_results = []
        with ThreadPoolExecutor(max_workers=15) as executor:
            future_to_url = {executor.submit(scrape_detail, link): link for link in links}
            
            for future in as_completed(future_to_url):
                data = future.result()
                processed_links += 1
                
                if data:
                    batch_results.append(data)
                    all_results.append(data)

                    # DỪNG NGAY KHI ĐỦ SỐ LƯỢNG
                    if len(all_results) >= target_count:
                        print(f"\nĐÃ ĐỦ {target_count} BÀI HỢP LỆ → DỪNG TOÀN BỘ!")
                        return all_results 

        print(f"Trang {page} hoàn thành: +{len(batch_results)} bài hợp lệ (tổng: {len(all_results)})")

        page += 1
        time.sleep(0.1)

    print(f"\nHOÀN TẤT! Thu thập được {len(all_results)} bài hợp lệ")
    return all_results


# -----------------------------
# 5) Chạy thử
# -----------------------------
if __name__ == "__main__":
    data = crawl_full()
    print("Tổng số bài crawl được:", len(data))
    print(data[:10])
    save_to_csv(data, "thuviennhadat_raw.csv")
